# UBDS 2026: Basic Python
## Day 3: Take control of your code / Візьміть контроль над своїм кодом

Main contributor: @vvsbiocode

Second contributor: ChatGPT & Codex

### Topics covered / Сьогоднішні теми
1. Loop comprehension / Скорочений запис циклу
2. Writing custom functions / Написання своїх функцій
3. Reading errors and debugging in Jupyter/ Читання і вирішення помилок у Jupyter
4. Good practices in Python project structure / Хороші звички в організації пітон-проєктів


We will use the `biopython` package to work with a codon table and pairwise sequence alignment. / 
Ми будемо використовувати пакет `biopython` для роботи з таблицею кодонів та попарного вирівнювання послідовностей.


# Import the tools we need

- `biopython` helps in processing biological sequences / `biopython` допомагає в обробці біологічних послідовностей
- `logging` helps handle messages a program wants the user to know about during execution / `logging` допомагає обробляти повідомлення, про які програма хоче, щоб користувач знав під час виконання
- `warnings` helps to raise custom warnings / `warnings` допомагає викликати спеціальні попередження
- `os` handles directories / `os` обробляє директорії


In [2]:
from Bio.Data import CodonTable
from Bio import pairwise2
from Bio.Align import substitution_matrices
from Bio.Data import IUPACData
import logging
import warnings
import os

/Users/iseult/gitlab/Basic-python-2026/Basic-python-2026/.venv/lib/python3.12/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


## PROBLEM 1: How to fit same amount of code into less lines? 
## /
## ПРОБЛЕМА 1: Як вмістити той самий обсяг коду в меншу кількість рядків?

### Loop comprehension / Cкорочений запис циклу

To illustrate the principle, we'll do a typical bioinformatics task:

Transform DNA sequence into a protein!

/

Щоб проілюструвати принцип, ми виконаємо типове завдання з біоінформатики:

Перетворіть послідовність ДНК на білок!


In [3]:
# Load a standard codon table
# Завантажити стандартну таблицю кодонів
standart_codon_table = CodonTable.unambiguous_dna_by_name["Standard"]

#### Q: What are a triplet and a codon table?
#### П: Що таке триплет і таблиця кодонів?


In [4]:
print(standart_codon_table)

Table 1 Standard, SGC0

  |  T      |  C      |  A      |  G      |
--+---------+---------+---------+---------+--
T | TTT F   | TCT S   | TAT Y   | TGT C   | T
T | TTC F   | TCC S   | TAC Y   | TGC C   | C
T | TTA L   | TCA S   | TAA Stop| TGA Stop| A
T | TTG L(s)| TCG S   | TAG Stop| TGG W   | G
--+---------+---------+---------+---------+--
C | CTT L   | CCT P   | CAT H   | CGT R   | T
C | CTC L   | CCC P   | CAC H   | CGC R   | C
C | CTA L   | CCA P   | CAA Q   | CGA R   | A
C | CTG L(s)| CCG P   | CAG Q   | CGG R   | G
--+---------+---------+---------+---------+--
A | ATT I   | ACT T   | AAT N   | AGT S   | T
A | ATC I   | ACC T   | AAC N   | AGC S   | C
A | ATA I   | ACA T   | AAA K   | AGA R   | A
A | ATG M(s)| ACG T   | AAG K   | AGG R   | G
--+---------+---------+---------+---------+--
G | GTT V   | GCT A   | GAT D   | GGT G   | T
G | GTC V   | GCC A   | GAC D   | GGC G   | C
G | GTA V   | GCA A   | GAA E   | GGA G   | A
G | GTG V   | GCG A   | GAG E   | GGG G   | G
--+---------

In [5]:
# Set a DNA sequence, which encodes a short protein
# Встановити послідовність ДНК, яка кодує короткий білок
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"

# Initialize a variable to store the protein sequence
# Ініціалізувати змінну для зберігання послідовності білка
protein1 = ""

# Loop through the sequence in steps of 3
# Пройдіть по послідовності кроками по 3
for i in range(0, len(dna_sequence), 3):

    # Select one consecutive codon
    # Виберіть один послідовний кодон
    codon = dna_sequence[i:i+3]

    # Retrieve the amino acid encoded by the codon. If the codon is atypical, insert "?"
    # Add amino acid to a protein string
    # Отримати амінокислоту, закодовану кодоном. Якщо кодон нетиповий, вставити "?"
    # Додати амінокислоту до білкового рядка
    protein1 += standart_codon_table.forward_table.get(codon, "?")

print(protein1)

MAIVMGR?KGAR?


In [7]:
# Three lines of code... can we do less?
# Три рядки коду... чи можемо зробити менше?

protein2 = "".join([standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") for i in range(0, len(dna_sequence), 3)])
print(protein2)

MAIVMGR?KGAR?


#### Q: Are protein1 and protein2 different? / Питання: Чи відрізняються protein1 та protein2?


In [8]:
print("Protein 1", protein1)
print("Protein 2", protein2)
print("Are they the same?", protein1==protein2)

Protein 1 MAIVMGR?KGAR?
Protein 2 MAIVMGR?KGAR?
Are they the same? True


**Let's unfold what just happened.**
**Давайте детально розглянемо, що щойно сталося.**

It is possible to write a loop in one line. 
Можна написати цикл в один рядок.

- Assign a list to a variable.
- Призначити список змінній.

```python
protein2 = []
```

- Include "for" statement.
- Додайте заяву «за».

```python
protein2 = [for i in range(0, len(dna_sequence), 3)]
```

- Add execution of the function before "for" statement. Note, how instead of creating `codon` variable, its value is passed directly to the list.
- Додати виконання функції перед оператором "for". Зауважте, що замість створення змінної `codon` її значення передається безпосередньо в список.

```python
protein2 = [
    standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") 
    for i in range(0, len(dna_sequence), 3)
    ]
```

- For our case, we have to add another trick. Now, `protein2` is a list of amino acids. To convert it into a string, pass the whole statement to the function `join()` with an empty string "" as the delimiter. This concatenates all string elements of the list.
- У нашому випадку нам потрібно додати ще один трюк. Тепер `protein2` – це список амінокислот. Щоб перетворити його на рядок, передайте весь код функції `join()` з порожнім рядком "" як роздільником. Це об'єднує всі рядкові елементи списку.

```python
protein2 = "".join([...])
```

_Of course, you do not have to follow the exact order of the steps. But the final result always looks the same._
_Звичайно, вам не потрібно дотримуватися точного порядку кроків для скороченого написання списку. Але кінцевий результат завжди виглядає однаковим._

Something does not make sense. / Дещо не складає сенсу.

Why does protein sequence contain unrecognized triplets? Maybe the DNA sequence is not divisible by 3? / Чому ця послідовність містить нерозпізнані триплети? Може послідовність ДНК не ділиться на 3?

We can incorporate a condition inside a loop comprehension to demand that all checked sequences are triplets. / Ми можемо включити умову всередину скороченого циклу, щоб вимагати, щоб усі перевірені послідовності були триплетами.

A condition statement always follows the `for` statement. / Умова завжди слідує за оператором `for`.


In [9]:
# Here we again unfold loop comprehension for clarity, but it can be written in one line
# Тут ми знову розгортаємо скорочений цикл для ясності, але це можна записати в один рядок

protein3 = "".join(
    [
        standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") 
        for i in range(0, len(dna_sequence), 3)
        if len(dna_sequence[i:i+3]) == 3 # condition / умова
    ]
)

#### Q: How an added condition makes protein3 different from protein2?
#### П: Чим додана умова відрізняє protein3 від protein2?


In [10]:
print("Protein 2 (w/o condition) length / Довжина білка 2 (без умови) : ", len(protein2))
print("Protein 3 (w/ condition) length / Довжина білка 3 (з умовою) : ", len(protein3))
print(protein2)
print(protein3)

Protein 2 (w/o condition) length / Довжина білка 2 (без умови) :  13
Protein 3 (w/ condition) length / Довжина білка 3 (з умовою) :  12
MAIVMGR?KGAR?
MAIVMGR?KGAR


There is still one unrecognised triplet in the protein3 sequence.

У послідовності protein3 все ще є один нерозпізнаний триплет.


#### Q: Does codon table include stop codons?
#### П: Чи містить таблиця кодонів стоп-кодони?


In [11]:
stop_codon = "TGA"
print(standart_codon_table.forward_table.get(stop_codon, "?"))

?


In [12]:
print(standart_codon_table.forward_table)

{'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L', 'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'TAT': 'Y', 'TAC': 'Y', 'TGT': 'C', 'TGC': 'C', 'TGG': 'W', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L', 'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P', 'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M', 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T', 'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R', 'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V', 'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A', 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E', 'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'}


We can add multiple conditions in a list comprehention. / 
Ми можемо додати кілька умов в скорочений список.

As in full loops, a loop comprehension condition can not only filter out the elements, but also control the output. / 
Як і в повних циклах, умова скороченого циклу може не лише фільтрувати елементи, але й керувати виводом.

Wrap the function call statement into `()` and insert a condition. Whatever has to be executed if the condition is fulfilled goes before `if` and second statement goes after `else`. / 
Оберніть оператор виклику функції в `()` та вставте умову. Те, що має бути виконано, якщо умова виконана, йде перед `if`, а другий оператор йде після `else`.


In [16]:
protein4 = "".join(
    [
        # Everything in brackets handles a single triplet 
        (
            standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") # (2.1) get an amino acid / (2.1) отримати амінокислоту
            if dna_sequence[i:i+3] not in standart_codon_table.stop_codons # (1) if codon is not a stop codon / (1) якщо кодон не є стоп-кодоном
            else
            "*" # (2.2) if not, return asteriks / (2.2) якщо ні, повернути зірочку
        )
        # Iterate through entire DNA sequence in steps of 3
        for i in range(0, len(dna_sequence), 3)
        # Check the extracted nucleotides are len = 3
        if len(dna_sequence[i:i+3]) == 3 
    ]
)

protein4

'MAIVMGR*KGAR'

In [17]:
print("Protein 3 (w/ one condition) length / Довжина білка 3 (без умови) : ", len(protein3))
print("Protein 4 (w/ two conditions) length / Довжина білка 4 (з умовою) : ", len(protein4))
print(protein3)
print(protein4)

Protein 3 (w/ one condition) length / Довжина білка 3 (без умови) :  12
Protein 4 (w/ two conditions) length / Довжина білка 4 (з умовою) :  12
MAIVMGR?KGAR
MAIVMGR*KGAR


From a biological perspective, Protein 4 does not make sense. The protein sequence has to stop after a stop codon. / З біологічної точки зору, Білок 4 не має сенсу. Послідовність білка має закінчуватися після стоп-кодона.

Note, you can also put several conditions one after another, where the second statement going after the first `else` is executed when the second `if` is fulfilled. / Зверніть увагу, що ви також можете поставити кілька умов одну за одною, де другий оператор, що йде після першого `else`, виконується, коли другий `if` є дійсним.


#### Q: Can we put a condition for a loop to stop inside a comprehension?
#### П: Чи можемо ми поставити умову для зупинки циклу всередині стислого списку?


In [18]:
# This nested condition tries to stop the loop when a stop codon is met
# Ця вкладена умова намагається зупинити цикл, коли зустрічається стоп-кодон
protein5 = "".join(
    [
        (
            # Within brackets we handle the triplet 
            standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?")
            if dna_sequence[i:i+3] not in standart_codon_table.stop_codons
            else 
            # Break = Stop loop when a stop codon is read 
            break 
            if dna_sequence[i:i+3] in standart_codon_table.stop_codons
            else
            ""
        )
        # Iterate in steps of 3 the DNA sequence
        for i in range(0, len(dna_sequence), 3)
        if len(dna_sequence[i:i+3]) == 3 
    ]
)

SyntaxError: invalid syntax (3672625682.py, line 9)

It is impossible to stop a loop in a comprehension until it runs out of things to loop through. Also, note how we had to write `dna_sequence[i:i+3]` four times. / Неможливо зупинити цикл в стислому списку, доки не закінчаться елементи списку. Також зверніть увагу, як нам довелося чотири рази написати `dna_sequence[i:i+3]`.

Loop in list comprehension is meant to shrink simple tasks into one line of code. / Цикли в стислому списку призначений для стиснення простих завдань до одного рядка коду.

To accommodate more complex tasks, either use full loops or write a custom function. / Для виконання складніших завдань використовуйте повні цикли або напишіть власну функцію.


### Mini-challenge: Break down a loop comprehension

In [ ]:
# For example...
for i in range(0, len(dna_sequence), 3):
    codon = dna_sequence[i:i+3]
    # Add the if statements for codon length = 3 and not a stop codon
    protein1 += standart_codon_table.forward_table.get(codon, "?")


### Mini-challenge: Dictionary comprehension
### Міні-завдання: Розуміння словника

Using `dna_sequence`, create a dictionary in a single line of code where the keys are RNA codons, and the values are the corresponding amino acids. Apply list comprehension principals on dictionary. / Використовуючи `dna_sequence`, створіть словник в одному рядку коду, де ключами є кодони РНК, а значеннями – відповідні амінокислоти. Застосуйте принципи стислого списку до словника.

In [ ]:
# write your code here
{
    key : value
    for item in iterable (DNA SEQUENCE)
    if condition is True
}

# Dictionary comprehension syntax:
# { key : value for item in iterable if condition }
#
#   1. Create the key       -> dna_sequence[i:i+3].replace("T", "U")
#   2. Create the value     -> standart_codon_table.forward_table.get(...)
#   3. Loop over the DNA    -> for i in range(0, len(dna_sequence), 3)
#   4. Ignore incomplete codons -> if len(dna_sequence[i:i+3]) == 3
#
# The .replace("T", "U") converts each DNA codon into its RNA equivalent.


In [19]:
####################
##### Solution #####
####################

codon_dictionary = {
    dna_sequence[i:i+3].replace("T", "U"):                                              # Dictionary key: RNA codon
    standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?")                    # Dictionary value: Amino acid ("?" if codon is unknown)
    for i in range(0, len(dna_sequence), 3)                                              # Loop through the sequence three bases at a time
    if len(dna_sequence[i:i+3]) == 3                                                     # Only include complete codons
}

# PROBLEM 2: I cannot find a package that does what I want.
# ПРОБЛЕМА 2: Я не можу знайти пакет, який виконує потрібні мені функції.


Similarly to how `biopython` functions work for general bioinformatics purposes, we can create our own functions that suit the specific needs of our analysis. / Подібно до того, як функції `biopython` працюють для загальних біоінформатичних цілей, ми можемо створювати власні функції, які відповідають конкретним потребам нашого аналізу.

Another advantage of writing a function - you do not need to repeat the same big chunk of code anymore. / Ще одна перевага написання функції - вам більше не потрібно повторювати один і той самий великий фрагмент коду з разу в раз.

Let's create a function that performs codon-to-amino acid conversion and is aware of the triplet condition and stop codons. Meet a custom function, `real_translation`! / Давайте створимо функцію, яка виконує перетворення кодонів в амінокислоти та знає про триплетну умову та стоп-кодони. Знайомтесь, це власна функція `real_translation`!

#### Q: What do we want a custom function to take as input and give as output?
#### П: Що ми хочемо, щоб власна функція приймала на вхід і видавала на вихід?

In [20]:
protein5 = "".join([real_translation(dna_sequence[i:i+3]) for i in range(0, len(dna_sequence), 3)])

NameError: name 'real_translation' is not defined

Similar to variables, a custom function has to be defined **before** execution. / 
Подібно до змінних, власну функцію потрібно визначити **перед** виконанням.

Custom functions can take nothing as input, one or more arguments. / Власні функції не можуть приймати нічого на вхід, один або декілька аргументів.

They can return or not return objects. / Вони можуть повертати об'єкти або не повертати.


In [21]:
# It is a good practice to describe inside the function what it does
# Рекомендується описувати всередині функції, що вона робить
def real_translation(dna_sequence):
    """
    This function translates dna_sequence in protein sequence.
    Ця функція перетворює dna_sequence у послідовність білка.
    Input:
        dna_sequence: string of nucleotides / рядок нуклеотидів
    Output:
        protein: string of amino acids / рядок амінокислот
    """
    protein = ""
    # Loop through the sequence in steps of 3
    # Перебираємо послідовність кроками по 3
    for i in range(0, len(dna_sequence), 3):
        codon = dna_sequence[i:i+3]
        
        # Make sure codon has length 3
        # Переконаємося, що кодон має довжину 3
        if len(codon) < 3:
            break
        # Stop the function when a stop codon was found
        # Зупинити функцію, коли знайдено стоп-кодон
        if codon in standart_codon_table.stop_codons:
            break
        else:
            # Retrieve the amino acid encoded by the codon. If the codon is atypical, insert "?"
            # Отримання амінокислоти, закодованої кодоном. Якщо кодон нетиповий, вставити "?"
            protein += standart_codon_table.forward_table.get(codon, "?")
    

Attempt N 1 / Спроба 1


In [22]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
real_translation()
print(protein)

TypeError: real_translation() missing 1 required positional argument: 'dna_sequence'

If a custom function was designed to receive arguments, it will demand them. / Якщо користувацька функція була розроблена для отримання аргументів, вона їх вимагатиме.


Attempt N 2 / Спроба 2


In [23]:
# function should return variable `protein` 
# функція повинна повертати змінну `protein`
real_translation(dna_sequence) 
print(protein)

NameError: name 'protein' is not defined

If a variable was defined inside a custom function - it is not accessible from outside. / Якщо змінна була визначена всередині користувацької функції, вона недоступна ззовні.

To make a variable accessible, add this to the end of the function `real_translation`. / Щоб зробити змінну доступною, додайте це в кінець функції `real_translation`.

```python
return protein
```

Note how below we define a target variable (`protein`) as storage for the `real_translation` output. / Зверніть увагу, як нижче ми визначаємо цільову змінну (`protein`) як сховище для виводу `real_translation`.


In [ ]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

It is useful to think about use cases of a function in advance and make is as generalised as needed. / Корисно заздалегідь продумати варіанти використання функції та зробити її настільки узагальненою, наскільки це необхідно.

Let's say that you want to use a different codon table for different parts of your analysis. **Usually** you use a standard one, but sometimes one from mitochondria. / Припустимо, ви хочете використовувати різні таблиці кодонів для різних частин вашого аналізу. **Зазвичай** ви використовуєте стандартну, але іноді й таблицю з мітохондрій.


In [26]:
# Find the codon table of any mitochondria / Знайдіть таблицю кодонів будь-якої мітохондрії
for id, table in CodonTable.unambiguous_dna_by_id.items():
    if "mitochondria" in table.names[0].lower():
        print(id, "-", table.names)

2 - ['Vertebrate Mitochondrial', 'SGC1']
3 - ['Yeast Mitochondrial', 'SGC2']
4 - ['Mold Mitochondrial', 'Protozoan Mitochondrial', 'Coelenterate Mitochondrial', 'Mycoplasma', 'Spiroplasma', 'SGC3']
5 - ['Invertebrate Mitochondrial', 'SGC4']
9 - ['Echinoderm Mitochondrial', 'Flatworm Mitochondrial', 'SGC8']
13 - ['Ascidian Mitochondrial', None]
14 - ['Alternative Flatworm Mitochondrial', None]
16 - ['Chlorophycean Mitochondrial', None]
21 - ['Trematode Mitochondrial', None]
22 - ['Scenedesmus obliquus Mitochondrial', None]
23 - ['Thraustochytrium Mitochondrial', None]
24 - ['Pterobranchia Mitochondrial', None]
33 - ['Cephalodiscidae Mitochondrial', None]


#### Q: Are the mitochondria codon table and the standard codon table different?
#### П: Чи відрізняються таблиця кодонів мітохондрій та стандартна таблиця кодонів?


In [27]:
CodonTable.unambiguous_dna_by_name["Standard"] == CodonTable.unambiguous_dna_by_name["Vertebrate Mitochondrial"]

False

In [28]:
standard = CodonTable.unambiguous_dna_by_name["Standard"]
mito = CodonTable.unambiguous_dna_by_name["Vertebrate Mitochondrial"]

# Compare codons that translate to amino acids
all_codons = set(standard.forward_table) | set(mito.forward_table)

for codon in sorted(all_codons):
    aa_standard = standard.forward_table.get(codon, "STOP")
    aa_mito = mito.forward_table.get(codon, "STOP")

    if aa_standard != aa_mito:
        print(f"{codon}: Standard = {aa_standard}, Mitochondrial = {aa_mito}")

AGA: Standard = R, Mitochondrial = STOP
AGG: Standard = R, Mitochondrial = STOP
ATA: Standard = I, Mitochondrial = M
TGA: Standard = STOP, Mitochondrial = W


To make our function applicable to any organism, we could pass a `codon_table_name` as an argument and it will be retrieved automatically. / Щоб зробити нашу функцію застосовною до будь-якого організму, ми можемо передати `codon_table_name` як аргумент, і він буде отриманий автоматично.

What if we want a standard codon table to be used by default? / Що робити, якщо ми хочемо, щоб стандартна таблиця кодонів використовувалася за замовчуванням?

To do so, give a value to the argument directly in function definition. / Для цього потрібно вказати значення аргументу безпосередньо у визначенні функції.
In this case, the user does not have to pass the `codon_table_name="Standard"` argument, only if a table different from "Standart" is required. / У цьому випадку користувачеві не потрібно передавати аргумент `codon_table_name="Standard"`, лише якщо потрібна таблиця, відмінна від "Standart".


### Q: Explain what is happening in the following function

In [30]:
def real_translation(dna_sequence, codon_table_name="Standard"):
    """
    This function translates dna_sequence in real protein sequence.
    Input:
        dna_sequence: string of nucleotides
        codon_table_name: string name
    Output:
        protein: string of amino acids
    """
    # Retrieve a codon table
    codon_table = CodonTable.unambiguous_dna_by_name[codon_table_name]
    protein = ""
    # Loop through the sequence in steps of 3
    for i in range(0, len(dna_sequence), 3):
        codon = dna_sequence[i:i+3]
        
        # Make sure codon has length 3
        if len(codon) < 3:
            break
        # Stop when a stop codon was found
        if codon in codon_table.stop_codons:
            break
        else:
            # Retrieve the amino acid encoded by the codon. If the codon is atypical, insert "?"
            protein += codon_table.forward_table.get(codon, "?")
    return protein

We can also define an argument by calling its name. / Ми також можемо визначити аргумент, викликавши його ім'я.

In [31]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(codon_table_name = "Vertebrate Mitochondrial", dna_sequence)
print(protein)

SyntaxError: positional argument follows keyword argument (4158634327.py, line 2)

Note, keyword arguments (with default value) have to be always called after positional arguments (without default). / Зверніть увагу, що ключові аргументи (зі значенням за замовчуванням) завжди мають викликатися після позиційних аргументів (без значення за замовчуванням).

#### Q: Does our function finally work?
#### П: Чи працює наша функція нарешті?

It is good practice to test a custom function on various scenarios. This way you will know if it works as expected. / Рекомендується тестувати власну функцію в різних сценаріях. Таким чином, ви дізнаєтеся, чи працює вона належним чином.


Test N 1 normal dna sequence / Тест N 1 ​​нормальна послідовність ДНК


In [32]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

MAIVMGR


Describe each of these test inputs. Are they failing a function? Why? / Опишіть кожен із цих тестових вхідних даних. Чи вони не відповідають дизайну функції?

Test N 2


In [33]:
dna_sequence = ["ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"]
protein = real_translation(dna_sequence)
print(protein)

Test N 3


In [34]:
dna_sequence = "AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGUA"
protein = real_translation(dna_sequence)
print(protein)

?A???GR?K?A?


Test N 4


In [36]:
dna_sequence = "AT"
protein = real_translation(dna_sequence)
print(protein)

Test N 5

In [37]:
dna_sequence = "GGGCTAGCCATTGTAATGGGCCGCAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

GLAIVMGRKGAR


Test N 6


In [38]:
dna_sequence = "TGATAA"
protein = real_translation(dna_sequence)
print(protein)

Test N 7


In [39]:
dna_sequence = "GGGCTAATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
codon_table_name = "Dragon"
protein = real_translation(dna_sequence, codon_table_name)
print(protein)

KeyError: 'Dragon'

Come up with your test / Придумайте свій тест

In [ ]:
# write your code here


We identified several problematic cases / Ми виявили кілька проблемних випадків:
- input is not a string / вхідні дані не є рядком
- non-DNA sequence / послідовність, що не є ДНК
- non-triplet sequence / послідовність, що не є триплетом
- sequence starting with either no start or stop codon / послідовність, що починається або без стартового, або без стоп-кодона
- sequence contains only non-coding triplets / послідовність містить лише некодувальні триплети
- unknown codon table name / невідома назва таблиці кодонів

To avoid these cases, we can warn a user (you or your colleagues) about weird input or program behavior. / Щоб уникнути цих випадків, ми можемо попередити користувача (вас або ваших колег) про дивні вхідні дані або поведінку програми у самій функції.

### What code can communicate to us? / Що код може нам сказати?

**INFO** -> "What is the program doing overall?" (for casual usage) / "Що робить програма загалом?" (для звичайного використання)

**DEBUG** -> "What exactly is happening step-by-step?" (for developers) / "Що саме відбувається крок за кроком?" (для розробників)

**WARNING** -> "Signal something is not ideal." (program continues) / "Сигнал про неідеальність." (програма продовжує роботу)

**EXCEPTION** -> "Signal something is wrong, but handleable." (program stops, you can correct) / "Сигнал про помилку, але це можна виправити." (програма зупиняється, ви можете виправити)

**ERROR** -> "Signal of code or system issues." (program stops, you cannot correct, unless you are a developer) / "Сигнал про проблеми з кодом або системою." (програма зупиняється, ви не можете виправити, якщо ви не розробник)


Info and debug messages can simply be written as print statements. / Інформаційні та налагоджувальні повідомлення можна просто виводити на екран.


In [40]:
# Info Example

def simple_translation(codon):
    print(f"Received codon {codon}")
    amino_acid = CodonTable.unambiguous_dna_by_name["Standard"].forward_table.get(codon, "?")
    return amino_acid

amino_acid1 = simple_translation("ATT")
print(amino_acid1)

Received codon ATT
I


Note how the warnings and exceptions are embedded in `if` statements, while info and debug are not necessarily. It makes sense to disturb a user or stop a program only in special conditions. / Зверніть увагу, як попередження та винятки вбудовані в оператори `if`, тоді як info та debug не обов'язково. Має сенс турбувати користувача або зупиняти програму лише за особливих умов.


For invoking warnings, use the `warnings` package. Note that first the function is executed and then the warning is printed out. / Для виклику попереджень використовуйте пакет `warnings`. Зверніть увагу, що спочатку виконується функція, а потім виводиться попередження.


In [41]:
# Warning Example

def simple_translation(codon):
    amino_acid = CodonTable.unambiguous_dna_by_name["Standard"].forward_table.get(codon, "?")
    if amino_acid == "?":
        warnings.warn(f"Non-DNA or non-coding triplet was given.")
    return amino_acid

amino_acid2 = simple_translation("AUA")
print(amino_acid2)

?


/var/folders/kl/s2thrg3x7xv8yn26zr1j_pd80000gn/T/ipykernel_50627/2848353416.py:4: UserWarning: Non-DNA or non-coding triplet was given.
  warnings.warn(f"Non-DNA or non-coding triplet was given.")


There are two ways to handle exceptions. / Існує два способи обробки винятків.

One, when you want to raise a custom exception before the code crashes. / По-перше, коли ви хочете викликати власний виняток до того, як код зламається.


In [42]:
# Exception Example 

def simple_translation(codon):
    if len(codon)<3:
        raise ValueError("Codon has to be 3 letters long.")
    # Operation occurs after value error
    amino_acid = CodonTable.unambiguous_dna_by_name["Standard"].forward_table.get(codon, "?")
    return amino_acid

amino_acid3 = simple_translation("TC")
print(amino_acid3)

ValueError: Codon has to be 3 letters long.

Two, when you want to work around exception that is raised by different opperation. / По-друге, коли ви хочете обійти виняток, викликаний іншою операцією.

Note, you can add several `return` statements in a custom function. / Зверніть увагу, що ви можете додати кілька операторів `return` у власній функції.

In [43]:
# Exception example 

def simple_translation(codon):
    try:
        amino_acid = CodonTable.unambiguous_dna_by_name["Standard"].forward_table.get(codon, "?")
        return amino_acid
    except TypeError:
        print("TypeError was caught. Codon is expected to be a string.")
        return None
    

simple_translation(["ATT"])

TypeError was caught. Codon is expected to be a string.


There are many types of exceptions. Each of their names tries to precisely describe what went wrong. / Існує багато типів винятків. Кожна з їхніх назв намагається точно описати, що пішло не так.

Often name = what went wrong + "Error" _(why not "Exception"? Idk, historical reasons...)_ / Часта назва = що пішло не так + "Error" _(чому не "Exception"? Не знаю, історичні причини...)_

If you do not understand meaning of exception - look it up on the Internet! / Якщо ви не розумієте значення винятку - знайдіть його в Інтернеті!

Here are some of the most often encountered / Ось деякі з найчастіше зустрічаються:

`NameError`	Variable name is not defined / Ім'я змінної не визначено

`IndexError`	List index out of range / Індекс списку поза діапазоном

`KeyError`	Dictionary key not found / Ключ словника не знайдено

`FileNotFoundError`	File doesn’t exist / Файл не існує 

`PermissionError`	No access to file or directory / Немає доступу до файлу або директорії

`ModuleNotFoundError` Module (imported package) doesn’t exist in your enviroment / Модуль (імпортований пакет) не існує у середовищі

`KeyboardInterrupt` Ctrl+C

`Exception` Generic exception, can mean anything / Загальний виняток, може означати що завгодно

etc. / і так далі.

You can also write your own exceptions! (not covered here) / Ви також можете написати власні винятки! (тут не розглядаються)


In [46]:
lst = [a, b, c, d]
print(lst[1])

NameError: name 'a' is not defined

In [47]:
lst = ['a', 'b', 'c', 'd']
print(lst[5])

IndexError: list index out of range

In [48]:
dct = {'a': 1, 'b': 2}
print(dct['c'])

KeyError: 'c'

### Logging

Logging is good practice when you develop a script, a tool, or a pipeline for internal usage. / Ведення журналу (логінг) – це гарна практика під час розробки скрипта, інструмента або конвеєра для внутрішнього використання.


In [50]:
# set up logging
logging.basicConfig(
    level=logging.INFO,
    # Format of the message
    format="%(asctime)s - %(levelname)s - %(message)s",
    # Format of the time
    datefmt="%H:%M:%S",
    # Forces print inside the notebook
    force=True 
)

Logging works similarly to print statements. But you can easily write the output to a `.log` file (not shown here). / Ведення журналу працює подібно до операторів друку. Але ви можете легко записати вивід у файл `.log` (тут не показано). 

Substitute `info` with other kinds of messages to apply logging for different purposes. / Замініть `info` іншими типами повідомлень, щоб застосувати ведення журналу для різних цілей.

In [51]:
logging.info("Welcome at Day 3 of python track!")
logging.warning("Did you eat today?!")

12:37:32 - INFO - Welcome at Day 3 of python track!
12:37:32 - WARNING - Did you eat today?!


It is unnecessary to combine logging of info, debug, and warnings with the above functions (`print` and `warnings.warn`), as they do not stop a program and would just duplicate a message. / Немає потреби поєднувати реєстрацію інформації, налагодження та попереджень з вищезазначеними функціями (`print` та `warnings.warn`), оскільки вони не зупиняють програму, а лише дублюють повідомлення.

However, `logging.exception` **cannot raise exception on its own**, so we combine it with `raise`. / Однак, `logging.exception` не може самостійно викликати виняток, тому ми поєднуємо його з `raise`.

Let's put it all into practice. / Давайте застосуємо все це на практиці.


#### Q: Any ideas on how to overcome issues in real_translation()?
/
#### П: Чи є якісь ідеї щодо того, як подолати проблеми в real_translation()?


This function looks huge, but only because it tries to be both super user- and developer-friendly. / Ця функція виглядає величезною, але лише тому, що вона намагається бути надзвичайно зручною як для користувача, так і для розробника.

*Any volunteers to explain the function?* / *Хтось може пояснити функцію?*


In [54]:
def real_translation(dna_sequence, codon_table_name="Standard"):
    """
    This function translates dna_sequence in real protein sequence.
    Input:
        dna_sequence: string of nucleotides
        codon_table_name: string name
    Output:
        protein: string of amino acids
    """
    logging.info(f"Start execution of real_translation()")

    # Check codon_table_name value
    logging.debug(f"Use codon table {codon_table_name}")
    table_names = list(CodonTable.unambiguous_dna_by_name.keys())
    if codon_table_name not in table_names:
        logging.exception(f"Codon table name is invalid. Try one of these: {table_names}")
        raise ValueError
    
    # Retrieve a codon table
    codon_table = CodonTable.unambiguous_dna_by_name[codon_table_name]
    logging.info(f"Successfully retrieved codon table.")

    # Check dna_sequence type
    logging.debug(f"Type of dna_sequence argument {type(dna_sequence)}")
    if not isinstance(dna_sequence, str):
        logging.exception(f"Argument dna_sequence type invalid. It should be a string (str).")
        raise TypeError
    
    # Check dna_sequence alphabet
    expected_alphabet = set("ATGC")
    sequence_alphabet = set(dna_sequence)
    logging.debug(f"String dna_sequence contains letters {sequence_alphabet}")
    if expected_alphabet != sequence_alphabet:
        logging.warning(f"Function real_translation() is suboptimal for non-DNA sequences. Use {expected_alphabet} alphabet.")
    
    # Check if dna_sequence contains at least one codon
    logging.debug(f"String dna_sequence legth {len(dna_sequence)}")
    if len(dna_sequence)<3:
        logging.exception(f"String dna_sequence is shorter than one codon. Minimal acceptable length is 3.")
        raise ValueError


    protein = ""
    codons = []
    cdna = 0 ; real_stop_codon = False
    # Loop through the sequence in steps of 3
    for i in range(0, len(dna_sequence), 3):

        codon = dna_sequence[i:i+3]

        # Make sure codon has length 3
        if len(codon) < 3:
            break

        # Check that sequence is coding from the start
        if i==0:
            cdna = 1 if codon == "ATG" else 0
            if not cdna:
                logging.warning(f"String dna_sequence starts as non-coding. Skipping non-coding triplets.")
                continue
        
        # Check that start codon was found
        if codon == "ATG":
            cdna +=1
            if cdna==1:
                logging.info(f"Start codon was found on position {i}.")
        
        if cdna>1 and not real_stop_codon:
            real_stop_codon = codon in codon_table.stop_codons
            if real_stop_codon:
                logging.info(f"Stop codon was found on position {i}.")
        
        # Recod unique codons
        if codon not in codons:
            codons.append(codon)

        # Stop when a stop codon was found
        if codon in codon_table.stop_codons:
            break
        else:
            # Retrieve the amino acid encoded by the codon. If the codon is atypical, insert "?"
            protein += codon_table.forward_table.get(codon, "?")

    if protein=="":
        logging.debug(f"String dna_sequence unique codons {codons}")
        logging.warning(f"Protein sequence is empty. Probably dna_sequence consisted only of non-coding triplets.")
    elif protein!="" and not real_stop_codon:
        logging.exception(f"Stop codon was never found. Corrupted coding sequence")
        raise ValueError

    return protein

In [55]:
dna_sequence = "AAAATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

12:40:16 - INFO - Start execution of real_translation()
12:40:16 - INFO - Successfully retrieved codon table.
12:40:16 - WARNING - String dna_sequence starts as non-coding. Skipping non-coding triplets.
12:40:16 - INFO - Start codon was found on position 3.
12:40:16 - INFO - Stop codon was found on position 24.


MAIVMGR


You just learned from the inside about the biggest enemy of programmers: bugs. / Ви щойно дізналися зсередини про найбільшого ворога програмістів: помилки.


# PROBLEM 3: These red messages drive me crazy!
# ПРОБЛЕМА 3: Ці червоні лінії зводять мене з розуму!

![QUOTE](Orson-Scott-Card-Quote.jpg)


Debugging is a process of solving (removing) bugs. It often feel like a brain teaser, and they consume most of the time of bioinformaticians. / Налагодження (дебагінг) – це процес вирішення (видалення) помилок. Це часто схоже на головоломку, і саме вони займають більшу частину часу біоінформатиків.

So you encounter one or multiple errors (red text). What's next? / Отже, ви стикаєтеся з однією або кількома помилками (червоний текст). Що далі?

**Your goal in the first place is not to get rid of errors, but to understand them.** / **Ваша мета в першу чергу – не позбутися помилок, а зрозуміти їх.**


#### Q: What is the difference between handling a bug in two solutions below?
/
#### П: Яка різниця між обробкою помилки у двох наведених нижче рішеннях?

Error

-> RNA sequence was passed instead of expected protein. / Замість очікуваного білка було передано послідовність РНК.


In [56]:
def align_sequences(seq1, seq2, matrix_name="BLOSUM62"):
    matrix = substitution_matrices.load(matrix_name)
    alignments = pairwise2.align.globalds(
        seq1,
        seq2,
        matrix,        
        -10,
        -0.5
    )
    best_alignment = alignments[0]
    return best_alignment

seq1 = "UUUUUUUU"
seq2 = "UUUUUUUU"
print("Are compared sequences the same?",seq1==seq2)
best_alignment = align_sequences(seq1, seq2)
print(best_alignment)

Are compared sequences the same? True


SystemError: <built-in function _make_score_matrix_fast> returned a result with an exception set

Solution N 1

-> The developer assumed that if the error arose from two identical sequences, then `pairwise2.align.globalds` probably cannot handle perfect alignments. / Розробник припустив, що якщо помилка виникла через дві однакові послідовності, то `pairwise2.align.globalds`, ймовірно, не зможе обробляти ідеальні вирівнювання.


In [59]:
def align_sequences(seq1, seq2, matrix_name="BLOSUM62"):
    matrix = substitution_matrices.load(matrix_name)
    # Try to align the sequences 
    try:
        alignments = pairwise2.align.globalds(
            seq1,
            seq2,
            matrix,        
            -10,
            -0.5
        )
        best_alignment = alignments[0]
        return best_alignment
    # The function cannot handle perfect alignment so we 
    except:
        print("perfect aligment")
        return f"Alignment(seqA={seq1}, seqB={seq2}, score=100.0, start=0, end={len(seq1)})"

seq1 = "UUUUUUUU"
seq2 = "UUUUUUUU"
print("Are compared sequences the same?",seq1==seq2)
best_alignment = align_sequences(seq1, seq2)
print(best_alignment)

Are compared sequences the same? True
perfect aligment
Alignment(seqA=UUUUUUUU, seqB=UUUUUUUU, score=100.0, start=0, end=8)


Solution N 2

-> The developer identified that the substitution matrix `"BLOSUM62"` concerns proteins, meaning that the `align_sequences()` function by default expects protein sequences as input. Thus, the developer passed a custom `matrix_name` value and made sure that all nucleotides are DNA and not RNA. / Розробник визначив, що матриця заміщення `"BLOSUM62"` стосується білків, тобто функція `align_sequences()` за замовчуванням очікує на вхід послідовності білків. Таким чином, розробник передав власне значення `matrix_name` та переконався, що всі нуклеотиди є ДНК, а не РНК.


In [60]:
def align_sequences(seq1, seq2, matrix_name="BLOSUM62"):
    # Retrieve all possible amino acids from library
    amino_acids = list(IUPACData.protein_letters)
    # Look at the alphabet of the inputted sequences
    alphabet = set(seq1+seq2)

    # If the matrix provided is for proteins but inputted sequence in nucleotides
    if matrix_name=="BLOSUM62" and any(s not in amino_acids for s in alphabet):
        # Suggest to user how to fix the exception
        raise ValueError("Non-protein sequence was passed. Use different matrix_name to align DNA.")
    
    matrix = substitution_matrices.load(matrix_name)

    
    alignments = pairwise2.align.globalds(
        seq1,
        seq2,
        matrix,        
        -10, # Gap opening penalty
        -0.5 # Gap extension penalty
    )
    best_alignment = alignments[0]
    return best_alignment

seq1 = "UUUUUUUU"
seq2 = "UUUUUUUU"

# Developer ensures that DNA sequence instead of RNA sequence is inputted 
seq1 = seq1.replace("U","T")
seq2 = seq2.replace("U","T")
print("Are compared sequences the same?",seq1==seq2)
best_alignment = align_sequences(seq1, seq2, "NUC.4.4")
print(best_alignment)

Are compared sequences the same? True
Alignment(seqA='TTTTTTTT', seqB='TTTTTTTT', score=40.0, start=0, end=8)


**You understand the bug through understanding what input function expects, what it does and what are its limitations.** / **Ви розумієте суть помилки, розуміючи, чого очікує функція на введення, що вона робить і які її обмеження.**


Common steps in debugging. / Типові кроки налагодження.

Reproduce -> Reduce -> Inspect -> Fix -> Verify / 
Відтворення -> Зменшення -> Перевірка -> Виправлення -> Перевірка

- Forsee the bugs. / Передбачте помилки.

    _What can go wrong with XYZ data input? Wrap such cases into exceptions._ / _Що може піти не так з введенням даних XYZ? Об’єднайте такі випадки у винятки._

- Read the error message carefully (traceback). / Уважно прочитайте повідомлення про помилку (traceback).

    _Pay attention to the error type, line number, and what variables are involved in the error._ / _Зверніть увагу на тип помилки, номер рядка та які змінні пов’язані з помилкою._

- Check if the bug is reproducible every time you run the code. / Перевіряйте, чи помилка відтворюється щоразу, коли ви запускаєте код.

    _It can happen that you accidentally used the wrong object with same name._ / _Може статися, що ви випадково використали неправильний об’єкт з тим самим іменем._

- Check if assumptions about your input or function output are correct. Write quick tests. / Перевірте, чи правильні припущення щодо ваших вхідних даних або виводу функції. Пишіть швидкі тести.

    _Especially in a long pipeline, you may lose the logic of data transformation and function design. If you assume that a variable is of type X or contains data in format Y, check by printing it out._ / _Особливо в довгому конвеєрі, ви можете втратити логіку перетворення даних та проектування функцій. Якщо ви припускаєте, що змінна має тип X або містить дані у форматі Y, перевірте це, вивівши її._

- Use a test set where you know the expected outcome. / Використовуйте тестовий набір, для якого вам відомий очікуваний результат.

- Reduce your code to the smallest failing case. Isolate failing part. / Зменште свій код до найменшого випадку невдачі. Виділіть частину, що невдала.

    _If you use big data objects or perform complicated operations, extract representative data point(s) and/or write seperately the smallest part of code that could be failing._ / _Якщо ви використовуєте великі об'єкти даних або виконуєте складні операції, видобудьте репрезентативні точки даних та/або окремо напишіть найменшу частину коду, яка може дати збій._

- Change one thing at a time. / Змінюйте по одному елементу за раз.

    _Test code after each code alteration. Do not rewrite everything from scratch (unless its the easiest way to fix the code)._ / _Тестуйте код після кожної зміни коду. Не переписуйте все з нуля (хіба що це найпростіший спосіб виправити код)._

- Trace the flow of code step-by-step. / Відстежуйте потік коду крок за кроком.

    _If it is unclear what code is failing, why, or if it is impossible or hard to isolate it._ / _Якщо незрозуміло, який код дає збій, чому, або якщо його неможливо чи важко ізолювати._




Modern IDEs usually have a "debugging mode," which lets you see all variable values without printing them. / Сучасні IDE зазвичай мають "режим налагодження", який дозволяє бачити всі значення змінних без їх виведення на екран.

Let's see how it works for Jupyter Notebook. / Давайте подивимося, як це працює для Jupyter Notebook.


In [63]:
# Two bugs are unanticipated: one throws an error and one is silent
def debug_me(seq1, seq2):

    matrix = substitution_matrices.load("NUC.4.4")

    # Align the sequences setting gap scores 
    alignment = pairwise2.align.globalds(seq1, seq2, matrix, -10, 0.5)[0]

    seqA, seqB, score, start, end = alignment

    # Count how many nucleotides match at the same position in the alignment
    matches = 0
    for a, b in zip(seqA, seqB):
        # Check that a is equal to b
        if a != b:
            matches +=1

    # Calulate the identity score 
    identity = matches / len(seqA)

    return identity

# Difference between sequences is in one nucleotide -> expect high identity
seq1 = "ATGCT"
seq2 = "ATGTT"
identity = debug_me(seq1, seq2)
print("Identity:", identity)

ValueError: Gap penalties should be non-positive.

In [71]:
# Test debugger here


### Debugger

####  **Option 1:** Use the built-in `%debug`

1. Run a cell that throws an exception. / Запустіть комірку, яка викликає виняток.
2. Immediately after the error, run / Відразу після помилки виконайте:

```python
%debug
```

This will launch an interactive post-mortem debugger. / Це запустить інтерактивний налагоджувач.

| Command      | Meaning              |
| ------------ | -------------------- |
| `n`          | Next line            |
| `s`          | Step into a function |
| `c`          | Continue to the end  |
| `p variable` | Print variable value |
| `q`          | Quit debugger        |

#### **Option 2:** Use `%pdb on` to automatically enter debugger

```python
%pdb on
```

* Now, anytime an exception occurs, Jupyter will automatically drop you into the debugger. / Тепер, щоразу, коли виникає виняток, Jupyter автоматично перекидатиме вас до налагоджувача.
* `%pdb off` disables automatic debugging / вимикає автоматичне налагодження

| Command | Meaning                        |
| ------- | ------------------------------ |
| `n`     | next line                      |
| `s`     | step into function             |
| `c`     | continue until next breakpoint |
| `l`     | list code around current line  |
| `p var` | print variable value           |
| `q`     | quit debugger                  |

#### **Option 3:** Use `pdb` manually inside code

You can set breakpoints anywhere / Ви можете встановити точки зупинки будь-де:

```python
import pdb

def my_function(x, y):
    pdb.set_trace()  # execution will pause here
    result = x + y
    return result

my_function(3, 5)
```

* The notebook cell will enter interactive debugging mode at the `set_trace()` line. / Комірка блокнота перейде в інтерактивний режим налагодження на рядку `set_trace()`.
* You can inspect variables, step through code, etc. / Ви можете перевіряти змінні, покроково проходити код тощо.

# EXERCISE 1: Debug race.
/
# ВПРАВА 1: Гонка налагодження.

Find and tackle as many bugs as possible in **15 minutes** in the function `align_sequences()`. Use debug mode, if possible. To get hints, scroll down. / Знайдіть та виправте якомога більше помилок за **15 хвилин** у функції `align_sequences()`. Використовуйте режим налагодження, якщо можливо. Щоб отримати підказки, прокрутіть униз.

In [72]:
def align_sequences(seq1, seq2, matrix_name="BLOSUM62"):
    """
    Perform pairwise alignment between two sequences.
    """

    logging.info("Starting alignment")

    if type(seq1) != str or type(seq2) != str:
        logging.warning("Sequences should be strings")

    matrix = substitution_matrices.load(matrix_name)

    gap_open = -10
    gap_extend = -0.5

    alignments = pairwise2.align.globalxx(
        seq1,
        seq2,
        matrix,        
        gap_open,
        gap_extend
    )

    best_alignment = alignments[0]

    aligned_seq1, aligned_seq2, score = best_alignment

    logging.info(f"Alignment score: {scores}")

    if score > "10":
        logging.info("High score alignment")

    return alignment

seq1 = "MTEYKLVVVG123"
seq2 = "MTEYKLVVVGA"

result = align_sequences(seq1, seq2)

print(result)

13:10:19 - INFO - Starting alignment


TypeError: globalxx takes exactly 2 argument (5 given)

**Unanticipated bugs in align_sequences()** / **Непередбачені помилки в align_sequences()**
- wrong variable type check (should allow str only) / неправильна перевірка типу змінної (має дозволяти лише str)
- not normalizing case, like... / неправильне завантаження матриці (пізніше неправильна назва змінної)...

```python
    seq1 = seq1.upper()
    seq2 = seq2.upper()
```

- wrong matrix loading (wrong variable name later) / помарка в назві змінної
- typo in variable name
- wrong function for matrix (should use globalds, matrix is ignored by globalxx) / неправильна функція для матриці (має використовувати globalds, матриця ігнорується globalxx)
- assumes at least one alignment exists / припускається, що існує принаймні одне вирівнювання
- wrong unpacking (structure is different) / неправильне розпакування (структура відрізняється)
- logging wrong variable / реєстрація неправильної змінної
- logic bug (score comparison always False) / логічна помилка (порівняння score завжди False)
- returning wrong variable / повернення неправильної змінної
- invalid input sequence (contains numbers) / недійсна вхідна послідовність (містить числа)


If you are done debugging, execute this line. / Якщо ви завершили налагодження, виконайте цей рядок.

![CUTE](debugging-gift.jpg)


# PROBLEM 4: I am lost in my code, help...
/
# ПРОБЛЕМА 4: Я загубився у своєму коді, допоможіть...

Does it sound familiar? (if not yet, time will come) / Знайомо звучить? (якщо ще ні, то час прийде.)

_Too long notebooks (like this one XD). Tired of scrolling. Takes ages to rerun. The kernel crashes due to memory overload._ / _Занадто довгі блокноти (як цей XD). Набридло гортати. Перезапуск триває довго. kernel аварійно завершує роботу через перевантаження пам'яті._

_Many notebooks with unclear names and purposes. Cannot find the right one._ / _Багато блокнотів з незрозумілими назвами та призначенням. Не можу знайти потрібний._

_Running out of creative names for distinct variables and custom functions. Start putting 1, 2, 3... at the end of the names._ / _Закінчуються креативні назви для різних змінних та користувацьких функцій. Почніть додавати 1, 2, 3... в кінці назв._

Follow best practices of project structure to never struggle again! / Дотримуйтесь найкращих практик структурування проектів, щоб більше ніколи не мати проблем!

In [73]:
# Print directory tree
def print_tree(startpath, prefix=""):
    files = sorted(os.listdir(startpath))
    for i, f in enumerate(files):
        path = os.path.join(startpath, f)
        connector = "├── " if i < len(files) - 1 else "└── "
        print(prefix + connector + f)
        if os.path.isdir(path):
            extension = "│   " if i < len(files) - 1 else "    "
            print_tree(path, prefix + extension)

# This is a typical basic structure
print_tree("./real_aligment")

├── .gitignore
├── README.md
├── data
│   └── raw
│       └── example_dna.fasta
├── environment.yml
├── notebooks
│   └── dna_translation_alignment.ipynb
├── requirements.txt
├── setup.py
├── src
│   ├── __init__.py
│   └── bioseq.py
└── tests
    └── test_bioseq.py


#### Discuss reasoning behind project structure, which is driven both by good practices and script aim. Analyse the role of each file. Run the tests and notebook.
/
#### Обговоріть логіку структури проєкту, яка зумовлена ​​як належними практиками, так і метою скрипта. Проаналізуйте роль кожного файлу. Виконайте tests та notebooke.